In [ ]:
# ── Robot connection ──────────────────────────────────────────────────────────
# Normal mode: running notebooks directly on the Reachy host → use 'localhost'.
# Remote mode: set REACHY_IP to the robot's IP address before connecting.
# REACHY_IP = REACHY_IP   # physical robot (remote)
REACHY_IP = "localhost"         # running on Reachy host (default)

In [1]:
from reachy_sdk import ReachySDK

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Connect to Reachy. If you're working directly on the robot, use

```python
host='localhost'
```

If not, just get Reachy's IP address with

```bash
$ ifconfig
```

and replace the host argument zith the IP address.

In [2]:
reachy = ReachySDK(host=REACHY_IP)

Check if you see the five joints of the head.

In [3]:
reachy.head

<Head joints=<Holder
	<Joint name="neck_roll" pos="10.37" mode="stiff">
	<Joint name="neck_pitch" pos="41.69" mode="stiff">
	<Joint name="neck_yaw" pos="-19.89" mode="stiff">
	<Joint name="l_antenna" pos="1.76" mode="stiff">
	<Joint name="r_antenna" pos="-118.07" mode="stiff">
>>

Use look_at to see if the zero has been actually done.

## Checking Orbita

In [8]:
reachy.turn_on('head')

Make Reachy look forward.

In [9]:
reachy.head.look_at(
    x=0.5,
    y=0,
    z=0,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

If this didn't work, it's likely that the zeros of Orbita have not been set. To do that use [orbita_zero.py](https://github.com/pollen-robotics/reachy_controllers/blob/master/setup/orbita_zero.py).

Next, reproduce the look_at sequence of the [documentation](https://pollen-robotics.github.io/reachy-2021-docs/sdk/first-moves/head/#orbita-look_at-method).

In [6]:
import time

look_right = reachy.head.look_at(
    x=0.5,
    y=-0.5,
    z=0.1,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

time.sleep(0.5)

look_down = reachy.head.look_at(
    x=0.5,
    y=0,
    z=-0.4,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

time.sleep(0.5)

look_left = reachy.head.look_at(
    x=0.5,
    y=0.3,
    z=-0.3,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

time.sleep(0.5)

look_front = reachy.head.look_at(
    x=0.5,
    y=0,
    z=0,
    duration=1.0,
    starting_positions={
        reachy.head.neck_roll: reachy.head.neck_roll.goal_position,
        reachy.head.neck_pitch: reachy.head.neck_pitch.goal_position,
        reachy.head.neck_yaw: reachy.head.neck_yaw.goal_position
    })

Make the head follow the end-effector. It is a good opportunity to check the forward kinematics of each arm as well.

**Press the square button 'interrupt the kernel' when you want to stop this.**

With the right arm:

In [7]:
try:
    x, y, z = reachy.r_arm.forward_kinematics()[:3,-1] # We want the translation part of Reachy's pose matrix
    reachy.head.look_at(x=x, y=y, z=z-0.05, duration=1.0) # There is a 5cm offset on the z axis

    time.sleep(0.5)

    while True:
        x, y, z = reachy.r_arm.forward_kinematics()[:3,-1]
        gp_dic = reachy.head._look_at(x, y, z - 0.05)
        reachy.head.neck_roll.goal_position = gp_dic[reachy.head.neck_roll]
        reachy.head.neck_pitch.goal_position = gp_dic[reachy.head.neck_pitch]
        reachy.head.neck_yaw.goal_position = gp_dic[reachy.head.neck_yaw]
        time.sleep(0.01)
except AttributeError:
    print('Reachy has no right arm!')

KeyboardInterrupt: 

With the left arm:

## Checking the antennas

Check that the l_antenna is actually the left and that r_antenna is the right.

In [ ]:
reachy.head.l_antenna.goal_position = -40

In [ ]:
reachy.head.l_antenna.goal_position = 0

In [ ]:
reachy.head.r_antenna.goal_position = -40

In [ ]:
reachy.head.r_antenna.goal_position = 0

Reproduce the sad and happy movements from the [documentation](https://pollen-robotics.github.io/reachy-2021-docs/sdk/first-moves/head/#antennas).

In [ ]:
def happy_antennas():
    reachy.head.l_antenna.speed_limit = 0.0
    reachy.head.r_antenna.speed_limit = 0.0
    
    for _ in range(9):
        reachy.head.l_antenna.goal_position = 10.0
        reachy.head.r_antenna.goal_position = -10.0

        time.sleep(0.1)

        reachy.head.l_antenna.goal_position = -10.0
        reachy.head.r_antenna.goal_position = 10.0

        time.sleep(0.1)
    
    reachy.head.l_antenna.goal_position = 0.0
    reachy.head.r_antenna.goal_position = 0.0
        
def sad_antennas():
    reachy.head.l_antenna.speed_limit = 70.0
    reachy.head.r_antenna.speed_limit = 70.0
    
    reachy.head.l_antenna.goal_position = 140.0
    reachy.head.r_antenna.goal_position = -140.0
    
    time.sleep(5.0)
    
    reachy.head.l_antenna.goal_position = 0.0
    reachy.head.r_antenna.goal_position = 0.0

In [ ]:
happy_antennas()

In [ ]:
sad_antennas()

Once you're done using the antennas, make Reachy look forward.

In [ ]:
reachy.head.look_at(x=0.5, y=0, z=0, duration=1.0)

and turn off the motors.

In [ ]:
reachy.turn_off('head')

## Get images

Check right and left image.

In [ ]:
# NOTE: camera not available in simulator — skip cells below on localhost
from PIL import Image

In [ ]:
Image.fromarray(reachy.left_camera.last_frame[:,:,::-1])

In [ ]:
Image.fromarray(reachy.right_camera.last_frame[:,:,::-1])